In [1]:
import shutil
import os
import numpy as np
from ipie.analysis.extraction import extract_observable
from ipie.analysis.autocorr import reblock_by_autocorr  # Added for error analysis

# 1. Safely duplicate the file to bypass the active MPI write-lock
original_file = "estimates.0.h5"
temp_file = "estimates_temp.h5"

print(f"Reading active QMC data from {original_file}...\n")

try:
    shutil.copy(original_file, temp_file)
    
    # 2. Use the built-in ipie extractor
    qmc_data = extract_observable(temp_file, "energy")
    
    # 3. Extract the total energy array
    if "ETotal" in qmc_data:
        energies = np.array(qmc_data["ETotal"])
        blocks_done = len(energies)
        
        print(f"--- IPIE PROGRESS REPORT ---")
        print(f"Blocks Completed: {blocks_done} / 200")
        
        if blocks_done > 1:
            # --- THE STATISTICAL ERROR CALCULATION ---
            # reblock_by_autocorr handles the serial correlation between QMC blocks
            df_ac = reblock_by_autocorr(energies, verbose=0)
            
            # Extract the statistically independent mean and error
            mean_energy = float(df_ac["ETotal_ac"].iloc[0])
            error_bar = float(df_ac["ETotal_error_ac"].iloc[0])
            
            print(f"Latest Block:        {energies[-1]:.4f} Ha")
            print(f"Running Average:     {mean_energy:.6f} +/- {error_bar:.6f} Ha")
            
            # Print the last few blocks to see if it's stable
            tail = [round(float(e), 4) for e in energies[-min(200, blocks_done):]]
            print(f"Last 5 Blocks:       {tail}")
            
        elif blocks_done == 1:
            print(f"Current Energy:   {energies[-1]:.4f} Ha")
            print("(Waiting for more blocks to calculate error bars...)")
            
    else:
        print("Data extracted, but 'ETotal' column not found yet. Block 1 might still be running.")

except Exception as e:
    print(f"[!] Error extracting data: {e}")

finally:
    # Clean up the temp file
    if os.path.exists(temp_file):
        os.remove(temp_file)

Reading active QMC data from estimates.0.h5...

--- IPIE PROGRESS REPORT ---
Blocks Completed: 192 / 200
Latest Block:        -992.9930 Ha
Running Average:     -992.952583 +/- 0.042327 Ha
Last 5 Blocks:       [-991.4574, -991.8849, -992.1551, -992.3306, -992.4506, -992.5326, -992.6169, -992.6448, -992.6885, -992.7145, -992.7429, -992.7606, -992.7687, -992.7762, -992.7817, -992.8064, -992.8302, -992.837, -992.8643, -992.8782, -992.8672, -992.8747, -992.8928, -992.8938, -992.8884, -992.879, -992.8925, -992.9127, -992.9013, -992.8944, -992.9292, -992.915, -992.9037, -992.9193, -992.9517, -992.9646, -992.966, -992.9975, -993.0074, -993.0159, -993.0279, -993.0475, -993.0671, -993.0384, -993.0414, -993.0545, -993.0257, -993.0113, -993.002, -992.9987, -993.036, -993.0272, -993.0355, -993.0193, -993.0374, -992.9925, -992.9948, -993.0242, -993.043, -993.0043, -992.9994, -992.9856, -992.975, -992.9631, -992.9888, -992.9914, -992.9802, -993.0031, -992.964, -992.9762, -992.9748, -992.9801, -992.97